[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/geolibre.ipynb)

# Visualize Vector and Raster Data with GeoLibre

`geolibre.Map` embeds the interactive [GeoLibre](https://geolibre.app) GIS app directly inside a Jupyter notebook cell, making it easy to inspect vector layers, Cloud Optimized GeoTIFFs (COGs), local GeoAI outputs, and swipe (split-map) comparisons.

| Feature | Method / Argument | Description |
| --- | --- | --- |
| Vector layers | `add_vector`, `add_geojson`, `add_choropleth` | Stream remote vector URLs or display local files and GeoDataFrames |
| Raster layers | `add_raster`, `add_cog` | Stream remote COGs or display local GeoTIFFs and xarray rasters |
| Split map | `split_map` | Interactive vertical or horizontal swipe slider between layer sets |
| Legends | `add_legend` | Built-in presets (such as `nlcd`, `esa_worldcover`) or custom mappings |
| UI chrome | `toolbar`, `layer_panel`, `style_panel`, `status_bar` | Show or hide individual UI panels via arguments, traits, or `GEOLIBRE_*` env vars |
| Renderers | `renderer`, `set_renderer`, `set_map_layout` | Switch between `maplibre`, `cesium` (3D globe), `mapbox`, and `arcgis`, or sync multi-pane grids |
| Export | `to_html`, `save_project` | Save a standalone HTML page or `.geolibre.json` project file |

In [ ]:
# %pip install -U geoai-py geolibre

In [ ]:
import os

import geoai
import geopandas as gpd
from geolibre import Map

cities_url = "https://data.source.coop/giswqs/opengeos/world_cities.geojson"
nlcd_url = "https://data.source.coop/giswqs/opengeos/nlcd_2021_land_cover_30m.tif"
naip_url = (
    "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip_rgb_train.tif"
)
buildings_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip_train_buildings.geojson"
masks_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip_building_masks.tif"

## Visualize vector data

The `full` layout displays the complete desktop chrome: top toolbar, Layers and Style panels, and the bottom status bar.

In [ ]:
m = Map(
    center=(-100, 40),
    zoom=3,
    app_layout="full",
    toolbar=True,
    layer_panel=True,
    style_panel=True,
    status_bar=True,
)
m.add_vector(cities_url, name="World Cities", circleRadius=3, fillColor="#ef4444")
m

## Visualize raster data (COG) with a legend

Hide the side panels to give a Cloud Optimized GeoTIFF the full width of the cell while keeping the top toolbar and a built-in NLCD land cover legend.

In [ ]:
m = Map(
    center=(-100, 40),
    zoom=4,
    app_layout="full",
    layer_panel=False,
    style_panel=False,
)
m.add_raster(nlcd_url, name="NLCD 2021")
m.add_legend(builtin="nlcd")
m

## Overlay choropleth vector footprints on aerial imagery

Compute geometric attributes (such as `area_m2`) with `geoai.add_geometric_properties` and overlay the graduated building footprints on top of the high-resolution NAIP raster, keeping only the Layers panel visible.

In [ ]:
buildings_gdf = geoai.add_geometric_properties(gpd.read_file(buildings_url)).to_crs(
    "EPSG:4326"
)

m = Map(
    center=(-117.598, 47.653),
    zoom=15,
    app_layout="full",
    toolbar=False,
    layer_panel=True,
    style_panel=False,
    status_bar=False,
)
m.add_raster(naip_url, name="NAIP Imagery")
m.add_choropleth(
    buildings_gdf,
    column="area_m2",
    name="Building Area (m²)",
    class_count=5,
    colormap="viridis",
    scheme="quantile",
)
m

## Create split-panel comparison maps

`split_map` adds an interactive Layer Swipe slider between two layer sets. Pass layer IDs returned by `add_raster` or `add_vector`, or `"__basemap__"` to compare against the basemap.

### Compare a raster layer with the basemap

In [ ]:
m_nlcd = Map(
    center=(-100, 40),
    zoom=4,
    basemap="liberty",
    app_layout="full",
    style_panel=False,
)
nlcd_id = m_nlcd.add_raster(nlcd_url, name="NLCD 2021")
m_nlcd.split_map(
    left_layers=nlcd_id,
    right_layers="__basemap__",
    orientation="vertical",
    position=50,
)
m_nlcd.add_legend(builtin="nlcd")
m_nlcd

### Compare segmentation masks with aerial imagery

In [ ]:
m_masks = Map(
    center=(-117.598, 47.653),
    zoom=15,
    app_layout="full",
    style_panel=False,
)
naip_id = m_masks.add_raster(naip_url, name="NAIP Imagery")
masks_id = m_masks.add_raster(
    masks_url,
    name="Building Masks",
    colormap="blues",
    rescale=[[0, 255]],
    opacity=0.7,
)
m_masks.split_map(
    left_layers=[naip_id, masks_id],
    right_layers=naip_id,
    orientation="vertical",
    position=50,
)
m_masks

### Compare vector polygons with aerial imagery (horizontal swipe)

In [ ]:
m_vec = Map(
    center=(-117.598, 47.653),
    zoom=15,
    app_layout="full",
    style_panel=False,
)
naip_id = m_vec.add_raster(naip_url, name="NAIP Imagery")
buildings_id = m_vec.add_vector(
    buildings_url,
    name="Buildings",
    fillColor="#ef4444",
    fillOpacity=0.45,
    strokeColor="#facc15",
    strokeWidth=1.5,
)
m_vec.split_map(
    left_layers=[naip_id, buildings_id],
    right_layers=naip_id,
    orientation="horizontal",
    position=50,
)
m_vec

## Change UI chrome after the map is shown

The UI options are live widget traits. Updating them reloads the map chrome while preserving every layer already loaded in `m`.

In [ ]:
m.style_panel = True
m.toolbar = True
m.status_bar = True
print(m.toolbar, m.layer_panel, m.style_panel, m.status_bar)

## Set defaults with environment variables

Environment variables (`GEOLIBRE_APP_LAYOUT`, `GEOLIBRE_THEME`, `GEOLIBRE_TOOLBAR`, `GEOLIBRE_LAYER_PANEL`, `GEOLIBRE_STYLE_PANEL`, `GEOLIBRE_STATUS_BAR`, `GEOLIBRE_RENDERER`) set defaults for every new `Map`, and explicit constructor arguments always take precedence.

In [ ]:
env_overrides = {
    "GEOLIBRE_APP_LAYOUT": "full",
    "GEOLIBRE_THEME": "dark",
    "GEOLIBRE_STYLE_PANEL": "hide",
    "GEOLIBRE_STATUS_BAR": "hide",
    "GEOLIBRE_LAYER_PANEL": "hide",
}
previous_env = {name: os.environ.get(name) for name in env_overrides}
try:
    os.environ.update(env_overrides)
    m_dark = Map(center=(-100, 40), zoom=3, layer_panel=True)
    m_dark.add_vector(
        cities_url, name="World Cities", circleRadius=3, fillColor="#facc15"
    )
finally:
    for name, previous in previous_env.items():
        if previous is None:
            os.environ.pop(name, None)
        else:
            os.environ[name] = previous
m_dark

## Switch rendering engines and linked multi-pane views

Render the same project with MapLibre (2D/3D globe), Cesium (3D globe), Mapbox, or the ArcGIS Maps SDK using `renderer=` or `set_renderer()`, or display synchronized multi-pane grids with `set_map_layout()`.

In [ ]:
m_3d = Map(
    center=(-100, 40),
    zoom=3,
    app_layout="full",
    renderer="cesium",
    style_panel=False,
)
m_3d.add_vector(cities_url, name="World Cities", circleRadius=3, fillColor="#ef4444")
m_3d

In [ ]:
print(m_3d.get_renderer())
m_3d.set_renderer("maplibre")
m_3d.set_projection("globe")

Show a synchronized 1x2 grid pairing a MapLibre map on the left with a Cesium 3D globe on the right.

In [ ]:
m_grid = Map(center=(-100, 40), zoom=3, app_layout="full", style_panel=False)
m_grid.add_raster(nlcd_url, name="NLCD 2021")
m_grid.set_map_layout(1, 2, view_kinds=["maplibre", "cesium"], sync_view=True)
m_grid

## Visualize local GeoAI raster and vector files

GeoLibre also works directly with local GeoTIFF and GeoJSON outputs produced by GeoAI workflows. Download sample files with `geoai.download_file` and pass the local paths to `add_raster` and `add_vector`.

In [ ]:
naip_path = geoai.download_file(naip_url)
buildings_path = geoai.download_file(buildings_url)

In [ ]:
m_local = Map(
    center=(-117.598, 47.653),
    zoom=15,
    app_layout="full",
    style_panel=False,
)
m_local.add_raster(naip_path, name="Local NAIP")
m_local.add_vector(
    buildings_path,
    name="Local Buildings",
    fillColor="#ef4444",
    fillOpacity=0.4,
    strokeColor="#facc15",
)
m_local

## Export to HTML and save a GeoLibre project

`to_html` writes a standalone HTML page carrying all layers, split-map settings, and UI chrome, while `save_project` saves a portable `.geolibre.json` file that can be reopened in Python or on [geolibre.app](https://geolibre.app).

In [ ]:
m_masks.to_html("geolibre_split_map.html", title="GeoAI + GeoLibre Split Map")
m_masks.save_project("geolibre_split_map.geolibre.json")